# Mastering NumPy for AI Engineering

## Learning Goal
- **What this notebook teaches**: NumPy arrays from first principles — dtype, shape, strides, vectorized operations, broadcasting, and linear algebra — the mathematical engine under every ML framework.
- **Background & links**: NumPy docs: https://numpy.org/doc/stable/ | NumPy for beginners: https://numpy.org/doc/stable/user/absolute_beginners.html
- **Why it matters in AI engineering**: PyTorch and TensorFlow tensors are conceptually NumPy arrays on GPU. Embeddings are NumPy arrays. Attention scores are matrix multiplications. Normalization is a vectorized operation. If you do not understand NumPy you cannot understand what your model is doing.

## Mental Model

```
Python list              NumPy array
─────────────────────    ───────────────────────────────────────────────
[1, 2, 3]                np.array([1, 2, 3])   dtype=int64  shape=(3,)

Heterogeneous types  →   Single dtype (all float32, all int64, ...)
Boxed Python objects →   Contiguous C memory block
O(n) element ops     →   O(1) SIMD vectorized ops (100-1000x faster)
No shape concept     →   shape=(rows, cols, depth, ...) — N-dimensional
```

**The key insight**: A NumPy array is a *view* into a contiguous block of memory.
Shape and dtype describe how to interpret that block.
`arr[i, j]` = `base_pointer + i * stride[0] + j * stride[1]`.

This is exactly what a neural network weight matrix is.

## Background Explanation

### Creating arrays

| Method | Result | Use |
|--------|--------|-----|
| `np.array([1,2,3])` | from Python list | Small arrays |
| `np.zeros((m, n))` | all zeros | Weight init |
| `np.ones((m, n))` | all ones | Mask init |
| `np.random.randn(m, n)` | standard normal | Random init |
| `np.arange(start, stop, step)` | range | Index arrays |
| `np.linspace(a, b, n)` | evenly spaced | Plotting / grids |

### Axis convention (critical)

```
arr.shape = (batch, seq_len, hidden_dim)
             axis=0  axis=1   axis=2

arr.mean(axis=0)  → mean over batch    → shape (seq_len, hidden_dim)
arr.mean(axis=1)  → mean over seq_len  → shape (batch, hidden_dim)
arr.mean(axis=-1) → mean over last dim → shape (batch, seq_len)
```

### dtype matters
- `float32` — standard for ML (GPU-friendly)
- `float64` — Python default, double precision, 2× memory
- `int64` — token IDs, indices
- `bool` — masks, attention masks

In [ ]:
import numpy as np
print("NumPy version:", np.__version__)

## Minimal Working Example — Embedding Similarity

In [ ]:
# Simulate two word embeddings (e.g. from word2vec / fastText)
np.random.seed(42)
emb_a = np.random.randn(768).astype(np.float32)   # "قطة" (cat)
emb_b = np.random.randn(768).astype(np.float32)   # "هرة" (cat, different Arabic word)
emb_c = np.random.randn(768).astype(np.float32)   # "سيارة" (car) — unrelated

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Dot product of L2-normalised vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"قطة  ↔  هرة  (both cat): {cosine_similarity(emb_a, emb_b):.4f}")
print(f"قطة  ↔  سيارة (cat/car): {cosine_similarity(emb_a, emb_c):.4f}")
print(f"Random embeddings are nearly orthogonal (≈0) — as expected")

## Exercise 1 — Batch Cosine Similarity Matrix

**Goal**: Compute pairwise cosine similarities between a matrix of query embeddings and a matrix of document embeddings — the core operation of every vector search system.

**Real context**: In RAG, you embed the query and all chunks, then find the k most similar chunks using cosine similarity.

In [ ]:
np.random.seed(0)
n_queries = 4
n_docs    = 8
dim       = 64   # small dimension for readability

Q = np.random.randn(n_queries, dim).astype(np.float32)  # (4, 64) query embeddings
D = np.random.randn(n_docs,    dim).astype(np.float32)  # (8, 64) doc embeddings

def batch_cosine_similarity(Q: np.ndarray, D: np.ndarray) -> np.ndarray:
    """
    Compute cosine similarity between every query and every document.

    Args:
        Q: (n_queries, dim) float32
        D: (n_docs, dim) float32

    Returns:
        sim: (n_queries, n_docs) float32
             sim[i, j] = cosine_similarity(Q[i], D[j])

    Constraint: do NOT use any Python loops. Use only NumPy operations.
    """
    # TODO 1: L2-normalise Q along dim axis → shape stays (n_queries, dim)
    # Hint: norm = np.linalg.norm(Q, axis=1, keepdims=True)

    # TODO 2: L2-normalise D along dim axis → shape stays (n_docs, dim)

    # TODO 3: Compute similarity matrix with matrix multiplication
    # Hint: normalised_Q @ normalised_D.T  →  shape (n_queries, n_docs)

    pass

sim = batch_cosine_similarity(Q, D)
print("Similarity matrix shape:", sim.shape)   # should be (4, 8)
print("Value range: [{:.3f}, {:.3f}]".format(sim.min(), sim.max()))

# For each query, find the top-2 most similar documents
top2 = np.argsort(sim, axis=1)[:, -2:][:, ::-1]
print("\nTop-2 doc indices per query:")
print(top2)

**Questions to think about**
1. Why do we normalise before multiplying instead of computing the full cosine formula element-by-element?
2. What is the computational complexity of this operation in terms of n_queries, n_docs, and dim?
3. What happens to the similarity matrix if two queries are identical?

## Solution 1

In [ ]:
def batch_cosine_similarity(Q: np.ndarray, D: np.ndarray) -> np.ndarray:
    Q_norm = Q / np.linalg.norm(Q, axis=1, keepdims=True)  # 1 — (n_q, dim)
    D_norm = D / np.linalg.norm(D, axis=1, keepdims=True)  # 2 — (n_d, dim)
    return Q_norm @ D_norm.T                                # 3 — (n_q, n_d)

sim = batch_cosine_similarity(Q, D)
print("Shape:", sim.shape)
print("Max per query:", sim.max(axis=1).round(4))

# Verify one entry matches the scalar function
manual = cosine_similarity(Q[0], D[0])
print(f"\nManual sim[0,0]={manual:.6f}  |  batch sim[0,0]={sim[0,0]:.6f}  | match={np.isclose(manual, sim[0,0])}")

## Exercise 2 — Attention Score Computation (Scaled Dot-Product)

**Goal**: Implement the core attention mechanism from the Transformer paper.

**Formula**: `Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) @ V`

This is literally what runs inside every Transformer layer.

In [ ]:
np.random.seed(7)
batch, seq_len, d_model = 2, 6, 32
d_k = d_model   # key dimension

Q = np.random.randn(batch, seq_len, d_k).astype(np.float32)   # queries
K = np.random.randn(batch, seq_len, d_k).astype(np.float32)   # keys
V = np.random.randn(batch, seq_len, d_model).astype(np.float32) # values

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax."""
    x_max = x.max(axis=axis, keepdims=True)    # subtract max for stability
    e_x   = np.exp(x - x_max)
    return e_x / e_x.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    """
    Args:
        Q: (batch, seq, d_k)
        K: (batch, seq, d_k)
        V: (batch, seq, d_model)
    Returns:
        output: (batch, seq, d_model)
        weights: (batch, seq, seq)   ← the attention weights
    """
    d_k = Q.shape[-1]

    # TODO 1: Compute raw scores = Q @ K^T, shape → (batch, seq, seq)
    # Hint: K needs to be transposed on its last two axes: K.transpose(0, 2, 1)
    scores = None

    # TODO 2: Scale by 1/sqrt(d_k)
    scores = None

    # TODO 3: Apply softmax over the last axis (key axis)
    weights = None

    # TODO 4: Weighted sum of values: weights @ V → shape (batch, seq, d_model)
    output = None

    return output, weights

output, weights = scaled_dot_product_attention(Q, K, V)
print("Output shape :", output.shape)    # (2, 6, 32)
print("Weights shape:", weights.shape)   # (2, 6, 6)
print("Weights row 0 sum:", weights[0, 0].sum().round(4))  # must be 1.0

## Solution 2

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores  = Q @ K.transpose(0, 2, 1)          # 1 — (batch, seq, seq)
    scores  = scores / np.sqrt(d_k)             # 2 — scale
    weights = softmax(scores, axis=-1)           # 3 — (batch, seq, seq)
    output  = weights @ V                        # 4 — (batch, seq, d_model)
    return output, weights

output, weights = scaled_dot_product_attention(Q, K, V)
print("Output shape:", output.shape)
print("Weights sum (should be 1.0):", weights[0, 0].sum().round(6))
print("\nAttention pattern for sequence 0, token 0:")
print(weights[0, 0].round(3))

## Common Mistakes

### Mistake 1 — Forgetting `axis` in reductions

In [ ]:
arr = np.array([[1., 2., 3.], [4., 5., 6.]])

print("No axis (global):", arr.mean())           # 3.5
print("axis=0 (per col):", arr.mean(axis=0))     # [2.5, 3.5, 4.5]
print("axis=1 (per row):", arr.mean(axis=1))     # [2.0, 5.0]

# In embeddings: you almost always want axis=-1 (per-vector)
embeddings = np.random.randn(100, 768)
norms = np.linalg.norm(embeddings, axis=-1)    # (100,) — one norm per embedding
print("\nNorm shape:", norms.shape)  # should be (100,), not a scalar

### Mistake 2 — Float dtype mismatches slow everything down

In [ ]:
a = np.ones((1000, 768), dtype=np.float32)
b = np.ones((1000, 768), dtype=np.float64)

# NumPy upcasts to float64 when mixing — GPU cannot use float64 efficiently
c = a + b
print("Mixed dtype result:", c.dtype)   # float64 — unintended!

# Always be explicit with dtype in ML code
c_correct = a + b.astype(np.float32)
print("Correct dtype:", c_correct.dtype)   # float32

### Mistake 3 — Views vs copies (silent aliasing bugs)

In [ ]:
original = np.array([1, 2, 3, 4, 5])
view     = original[1:4]      # this is a VIEW, not a copy
copy     = original[1:4].copy()

view[0] = 99
print("original after modifying view:", original)  # [1, 99, 3, 4, 5] — CHANGED!

copy[0] = 77
print("original after modifying copy:", original)  # unchanged — SAFE

## Real AI Engineering Usage

```python
import numpy as np

# Embedding normalisation (before vector DB insertion)
embeddings = model.encode(texts)                         # (n, dim) float32
norms      = np.linalg.norm(embeddings, axis=1, keepdims=True)
normalised = embeddings / norms                          # cosine-ready

# Batch dot-product for retrieval (the inner loop of every RAG system)
scores = query_emb @ doc_embs.T                          # (n_queries, n_docs)
top_k  = np.argsort(scores, axis=1)[:, -k:][:, ::-1]   # (n_queries, k)

# Softmax temperature scaling (LLM sampling)
logits  = model.forward(tokens)                          # (vocab_size,)
scaled  = logits / temperature
probs   = np.exp(scaled) / np.exp(scaled).sum()

# Padding + masking a batch of variable-length sequences
max_len   = max(len(s) for s in sequences)
padded    = np.zeros((len(sequences), max_len), dtype=np.int64)
attn_mask = np.zeros_like(padded, dtype=bool)
for i, seq in enumerate(sequences):
    padded[i, :len(seq)]    = seq
    attn_mask[i, :len(seq)] = True

# Precision/recall from confusion matrix
TP = np.diag(conf_matrix)
precision = TP / conf_matrix.sum(axis=0)
recall    = TP / conf_matrix.sum(axis=1)
```

## Final Mini Exercise — L2-Normalize a Batch and Verify

Given a matrix of raw embeddings `E` of shape `(n, dim)`:
1. L2-normalise each row so every vector has unit length.
2. Confirm that `normalised @ normalised.T` gives a matrix whose diagonal is all 1.0.
3. Find the top-3 most similar pairs (excluding self-similarity on the diagonal).

In [ ]:
np.random.seed(99)
E = np.random.randn(10, 32).astype(np.float32)

# TODO: implement the three tasks above
# 1 — normalise
# 2 — verify diagonal
# 3 — find top-3 pairs (hint: np.triu to look at upper triangle only, np.unravel_index)

## Key Takeaways

- NumPy arrays are **contiguous C memory blocks** — single dtype, vectorized, 100-1000× faster than Python loops.
- `shape`, `dtype`, and `axis` are the three things to always check first when debugging.
- **Never loop over elements** — express operations as matrix multiplications, broadcasts, and reductions.
- `float32` is the ML default. Mixing `float32` and `float64` silently produces `float64` (slow on GPU).
- Slices return **views** (no copy). `.copy()` when you need independence.
- Scaled dot-product attention, cosine similarity, and softmax are all pure NumPy operations — understand them before using `torch.nn.MultiheadAttention`.
- In PyTorch: `tensor.numpy()` ↔ `torch.from_numpy(arr)` — they share memory, no copy.